# Leakage detective: random, group и time split

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Задача

В таблице несколько событий на одного пользователя, а target зависит от устойчивого пользовательского эффекта и времени. Сравните random split, `GroupShuffleSplit` по user_id и временную границу. Объясните, почему оценки отличаются.

**Deliverable:** таблица из трёх ROC-AUC, пересечение user_id между train/test и рекомендация для production validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

rng = np.random.default_rng(42)
n_users, events_per_user = 250, 8
user_id = np.repeat(np.arange(n_users), events_per_user)
timestamp = np.tile(np.arange(events_per_user), n_users)
user_effect = rng.normal(size=n_users)[user_id]
x = user_effect + 0.15 * timestamp + rng.normal(scale=.7, size=len(user_id))
y = (user_effect + 0.08 * timestamp + rng.normal(size=len(user_id)) > 0).astype(int)
df = pd.DataFrame({"user_id": user_id, "timestamp": timestamp, "x": x, "target": y})


In [ ]:
def fit_score(train_idx, test_idx):
    # TODO: обучите LogisticRegression на столбце x и верните ROC-AUC.
    raise NotImplementedError

# TODO: сформируйте три пары индексов: random, group, time.
splits = {}


In [ ]:
# Self-check.
assert set(splits) == {"random", "group", "time"}
rows = []
for name, (train_idx, test_idx) in splits.items():
    overlap = len(set(df.iloc[train_idx].user_id) & set(df.iloc[test_idx].user_id))
    rows.append({"split": name, "roc_auc": fit_score(train_idx, test_idx), "user_overlap": overlap})
report = pd.DataFrame(rows)
assert report.roc_auc.between(0, 1).all()
assert report.loc[report.split == "group", "user_overlap"].item() == 0
report


## Вывод

Опишите единицу независимого наблюдения, production-сценарий и возможный gap. Высокая метрика random split сама по себе не доказывает leakage; покажите конкретный канал пересечения.